## Part 1


### **1. The Core Intuition Behind Contrastive Decoding**

The core intuition behind contrastive decoding is to **improve text quality by preventing the model from choosing generic, high-frequency words**. It achieves this by using two models: a powerful "expert" model (e.g., GPT-2 XL) and a smaller, weaker "amateur" model (e.g., GPT-2 Small). The amateur model is more likely to predict common, obvious words. By subtracting the amateur model's predictions from the expert's, we effectively penalize these generic tokens, forcing the expert to choose more specific, interesting, and contextually relevant words.

**How it differs from other strategies:**
* **Greedy/Beam Search:** These methods simply pick the token(s) with the highest probability according to a single model. They often get stuck in repetitive loops (e.g., "I think I think I think...") because such phrases are statistically likely.
* **Nucleus Sampling/Temperature:** These methods introduce randomness to increase diversity by sampling from a wider range of possible tokens. However, they don't actively penalize genericness; they just make boring outputs less likely to be chosen.
* **Contrastive Decoding's Uniqueness:** It is distinct because it uses a **second model as a guide** to reshape the probability distribution itself, actively steering the generation away from undesirable (dull) outputs rather than just randomly sampling from a better part of the distribution.

The main problem it solves is the tendency of language models to produce **dull, repetitive, and uninformative text**, a common failure mode where they default to statistically safe but uninteresting phrases.



### **2. The Mathematical Formulation and its Weaknesses**

The mathematical formulation of the contrastive objective is to find the next token, $x_t$, that maximizes the difference between the expert model's probability and the amateur model's probability. In practice, this is done at the logit level before the softmax function:

$$
\text{logit}_{\text{CD}} = \text{logit}_{\text{expert}} - \alpha \cdot \text{logit}_{\text{amateur}}
$$

The next token is then chosen by taking the `argmax` of this new contrastive logit, $\text{logit}_{\text{CD}}$.

Unconstrained maximization of this objective can lead to two major issues:

1.  **False Positives:** A token might be selected simply because the amateur model assigned it an extremely low probability, even if the expert model also considered it unlikely. This can cause the model to generate nonsensical or rare words that are out of context, creating a "false positive" which is a bad token that the objective function mistakenly scores highly.
2.  **False Negatives:** A perfectly valid and necessary token (like "the" or "is") might be unfairly penalized and ignored because the amateur model also assigns it a high probability. The objective function sees it as "too generic" and discards it, even when it's the most coherent choice. This is a "false negative" which is a good token that gets incorrectly rejected.



### **3. The Adaptive Plausibility Constraint and the Role of α**

The **adaptive plausibility constraint** is necessary to solve the problem of false positives and false negatives. Instead of applying the contrastive formula to all possible tokens in the vocabulary, it first creates a smaller "candidate set" of tokens that the **expert model already considers plausible**. This is typically done by filtering for all tokens whose probability, according to the expert, is above a certain threshold (e.g., all tokens in the top-k or top-p nucleus).

By doing this, we ensure two things:
* We eliminate **false positives** because any nonsensical token the expert model dislikes will never make it to the candidate set.
* We mitigate **false negatives** because if a common but necessary word is highly probable (plausible), it will be included in the candidate set and can still be chosen if it has the best contrastive score among the other plausible options.

The hyperparameter **α (alpha)** controls the **strength of the penalty** from the amateur model and directly manages the trade-off between coherence and diversity:

* **Low α:** A small α (e.g., 0.1) results in a weak penalty. The generated text will be closer to the expert model's original output.
* **High α:** A large α (e.g., 0.8) applies a strong penalty to common words. This pushes the model toward a surprising vocabulary, but if α is too high, the text can become incoherent or ungrammatical.

## Part 2

In [1]:
!git clone https://github.com/XiangLi1999/ContrastiveDecoding.git
%cd ContrastiveDecoding

Cloning into 'ContrastiveDecoding'...
remote: Enumerating objects: 1648, done.
remote: Counting objects: 100% (18/18), done.
remote: Compressing objects: 100% (15/15), done.
remote: Total 1648 (delta 2), reused 12 (delta 1), pack-reused 1630 (from 1)
Receiving objects: 100% (1648/1648), 259.49 MiB | 29.56 MiB/s, done.
Resolving deltas: 100% (489/489), done.
Updating files: 100% (1349/1349), done.
/content/ContrastiveDecoding


In [2]:
!pip install torch
!pip install transformers
!pip install datasets

In [3]:
from datasets import load_dataset

# Load the Wikitext-103 dataset
wikitext_dataset = load_dataset('wikitext', 'wikitext-103-raw-v1')

# You can inspect the dataset to see its structure
print(wikitext_dataset)

# You can also access the splits (train, test, validation) like this:
train_dataset = wikitext_dataset['train']
test_dataset = wikitext_dataset['test']
validation_dataset = wikitext_dataset['validation']

# And view a sample from the training data
print(train_dataset[0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

wikitext-103-raw-v1/test-00000-of-00001.(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-103-raw-v1/train-00000-of-00002(…):   0%|          | 0.00/157M [00:00<?, ?B/s]

wikitext-103-raw-v1/train-00001-of-00002(…):   0%|          | 0.00/157M [00:00<?, ?B/s]

wikitext-103-raw-v1/validation-00000-of-(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/1801350 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

DatasetDict({
    test: Dataset({
        features: ['text'],
        num_rows: 4358
    })
    train: Dataset({
        features: ['text'],
        num_rows: 1801350
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 3760
    })
})
{'text': ''}


In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Set the device to GPU if available, otherwise CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained('gpt2-xl')
# Add a padding token if it doesn't exist
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({'pad_token': '[PAD]'})

# Load the expert model (e.g., GPT-2 XL)
print("Loading expert model (gpt2-xl)...")
expert_model = AutoModelForCausalLM.from_pretrained('gpt2-xl').to(device)
expert_model.config.pad_token_id = tokenizer.pad_token_id

# Load the amateur model (e.g., GPT-2 Small)
print("Loading amateur model (gpt2)...")
amateur_model = AutoModelForCausalLM.from_pretrained('gpt2').to(device)
amateur_model.config.pad_token_id = tokenizer.pad_token_id

print("Models loaded successfully!")

Using device: cuda


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/689 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

Loading amateur model (gpt2)...


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Models loaded successfully!


In [5]:
def generate_contrastive(
    prompt,
    expert_model,
    amateur_model,
    tokenizer,
    alpha=0.1,
    amateur_temp=1.0,
    context_window=None,
    max_new_tokens=50
):
    """
    Generates text using contrastive decoding.
    """
    # Encode the input prompt
    input_ids = tokenizer.encode(prompt, return_tensors='pt').to(device)

    # Generation loop
    for _ in range(max_new_tokens):
        # Determine the context for the amateur model
        if context_window is not None:
            amateur_input_ids = input_ids[:, -context_window:]
        else:
            amateur_input_ids = input_ids

        # Get logits from both models
        with torch.no_grad():
            expert_logits = expert_model(input_ids).logits[:, -1, :]
            amateur_logits = amateur_model(amateur_input_ids).logits[:, -1, :]

        # Apply temperature to the amateur model's logits
        amateur_logits = amateur_logits / amateur_temp

        # Contrastive decoding formula
        final_logits = expert_logits - (alpha * amateur_logits)

        # Get the token with the highest probability (greedy decoding)
        next_token_id = torch.argmax(final_logits, dim=-1).unsqueeze(0)

        # Append the new token to the input sequence
        input_ids = torch.cat([input_ids, next_token_id], dim=-1)

        # Stop if we generate the end-of-sentence token
        if next_token_id.item() == tokenizer.eos_token_id:
            break

    # Decode and return the generated text
    return tokenizer.decode(input_ids[0], skip_special_tokens=True)

# Define a sample prompt for generation
prompt = "In a shocking finding, scientists discovered a herd of unicorns living in a remote, previously unexplored valley, in the Andes Mountains."

In [6]:
temperatures_to_test = [0.5, 1.0, 1.5]

print("--- Ablation Study: Varying Amateur Model Temperature ---")
for temp in temperatures_to_test:
    print(f"\n>> Generating with amateur temperature = {temp}")
    generated_text = generate_contrastive(
        prompt,
        expert_model,
        amateur_model,
        tokenizer,
        amateur_temp=temp
    )
    print(generated_text)
    print("-" * 50)

--- Ablation Study: Varying Amateur Model Temperature ---

>> Generating with amateur temperature = 0.5
In a shocking finding, scientists discovered a herd of unicorns living in a remote, previously unexplored valley, in the Andes Mountains.

The herd, which is estimated to number around 1,000 animals, is believed to have been there for at least 1,000 years.

The discovery was made by a team of scientists from the University of Bristol, led by Dr
--------------------------------------------------

>> Generating with amateur temperature = 1.0
In a shocking finding, scientists discovered a herd of unicorns living in a remote, previously unexplored valley, in the Andes Mountains.

The discovery was made by a team of scientists from the University of Bristol, the University of Oxford, and the University of Lausanne.

The team, led by Dr. David Evans, discovered the herd of unicorns in the
--------------------------------------------------

>> Generating with amateur temperature = 1.5
In a 

In [7]:
# Get the maximum context window size from the model's configuration
max_context = expert_model.config.n_positions  # For GPT-2, this is 1024
context_windows_to_test = [max_context, max_context // 2, 1]

print("\n--- Ablation Study: Restricting Amateur Model Context Window ---")
for window in context_windows_to_test:
    print(f"\n>> Generating with amateur context window = {window}")
    generated_text = generate_contrastive(
        prompt,
        expert_model,
        amateur_model,
        tokenizer,
        context_window=window
    )
    print(generated_text)
    print("-" * 50)


--- Ablation Study: Restricting Amateur Model Context Window ---

>> Generating with amateur context window = 1024
In a shocking finding, scientists discovered a herd of unicorns living in a remote, previously unexplored valley, in the Andes Mountains.

The discovery was made by a team of scientists from the University of Bristol, the University of Oxford, and the University of Lausanne.

The team, led by Dr. David Evans, discovered the herd of unicorns in the
--------------------------------------------------

>> Generating with amateur context window = 512
In a shocking finding, scientists discovered a herd of unicorns living in a remote, previously unexplored valley, in the Andes Mountains.

The discovery was made by a team of scientists from the University of Bristol, the University of Oxford, and the University of Lausanne.

The team, led by Dr. David Evans, discovered the herd of unicorns in the
--------------------------------------------------

>> Generating with amateur conte

In [8]:
!pip install mauve-text  # For MAUVE score
!pip install nltk             # For distinct-n scores
!pip install pandas           # For formatting the results table

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 16.0 MB/s eta 0:00:00


In [9]:
import torch
import nltk
import mauve
import numpy as np
from nltk.util import ngrams
from transformers import AutoModelForCausalLM, AutoTokenizer

nltk.download('punkt')
nltk.download('punkt_tab') # <-- fixed the error

# --- 1. Diversity (distinct-n) ---
def calculate_distinct_n(texts, n):
    """Calculates distinct-n score for a list of texts."""
    if not texts:
        return 0.0
    total_ngrams = 0
    unique_ngrams = set()
    for text in texts:
        tokens = nltk.word_tokenize(text.lower())
        for ngram in ngrams(tokens, n):
            unique_ngrams.add(ngram)
            total_ngrams += 1
    return len(unique_ngrams) / total_ngrams if total_ngrams > 0 else 0.0

# --- 2. Coherence (Perplexity) ---
# Use a separate, smaller model for efficient perplexity calculation
perplexity_model = AutoModelForCausalLM.from_pretrained('distilgpt2').to("cuda")
perplexity_tokenizer = AutoTokenizer.from_pretrained('distilgpt2')

def calculate_perplexity(texts, model, tokenizer):
    """Calculates the average perplexity for a list of texts."""
    total_perplexity = 0
    count = 0
    for text in texts:
        if not text:
            continue
        encodings = tokenizer(text, return_tensors='pt')
        input_ids = encodings.input_ids.to("cuda")

        with torch.no_grad():
            outputs = model(input_ids, labels=input_ids)
            neg_log_likelihood = outputs.loss

        perplexity = torch.exp(neg_log_likelihood)
        total_perplexity += perplexity.item()
        count += 1

    return total_perplexity / count if count > 0 else float('inf')


# --- 3. MAUVE Score ---
def calculate_mauve_score(generated_texts, human_texts):
    """Computes the MAUVE score."""
    # mauve.compute_mauve expects raw text lists
    mauve_result = mauve.compute_mauve(
        p_text=human_texts,
        q_text=generated_texts,
        device_id=0, # Use GPU
        verbose=False,
        featurize_model_name="gpt2-large" # As recommended by the MAUVE paper
    )
    return mauve_result.mauve

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [10]:
import pandas as pd
from datasets import load_dataset

# Load your expert and amateur models (assuming from previous step)
# device, tokenizer, expert_model, amateur_model should already be loaded.

# 1. Load Wikitext-103 and prepare prompts
print("Loading Wikitext-103 dataset...")
wikitext_test = load_dataset('wikitext', 'wikitext-103-raw-v1', split='test')
# Filter out empty or very short texts and create prompts
prompts = [text for text in wikitext_test['text'] if 50 < len(text) < 200]
num_samples = 50 # Using a small number for this example
prompts = prompts[:num_samples]
print(f"Using {len(prompts)} prompts for generation.")


# 2. Define experimental configurations
configurations = []
temperatures = [0.5, 1.0, 1.5]
max_context = amateur_model.config.n_positions
context_windows = [max_context, max_context // 2, 1]

for temp in temperatures:
    for window in context_windows:
        configurations.append({'temp': temp, 'window': window})

# 3. Run experiments and store results
results = []
for config in configurations:
    temp = config['temp']
    window = config['window']
    print(f"\n--- Running config: Temp={temp}, Window={window} ---")

    generated_texts = []
    for prompt in prompts:
        # Re-use the generate_contrastive function from the previous step
        gen_text = generate_contrastive(
            prompt, expert_model, amateur_model, tokenizer,
            amateur_temp=temp,
            context_window=window,
            max_new_tokens=60 # Generate a reasonable continuation
        )
        generated_texts.append(gen_text)

    # Evaluate metrics
    distinct_1 = calculate_distinct_n(generated_texts, 1)
    distinct_2 = calculate_distinct_n(generated_texts, 2)
    mauve_score = calculate_mauve_score(generated_texts, prompts)
    perplexity = calculate_perplexity(generated_texts, perplexity_model, perplexity_tokenizer)

    results.append({
        'Temp': temp,
        'Context Window': window,
        'Distinct-1': distinct_1,
        'Distinct-2': distinct_2,
        'MAUVE': mauve_score,
        'Perplexity': perplexity
    })

# 4. Present results in a table
df_results = pd.DataFrame(results)
df_results = df_results.round(4) # Round for clarity
print("\n--- Final Results ---")
print(df_results.to_string())

Loading Wikitext-103 dataset...
Using 50 prompts for generation.

--- Running config: Temp=0.5, Window=1024 ---


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/666 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

Featurizing q:   0%|          | 0/50 [00:00<?, ?it/s]


--- Running config: Temp=1.5, Window=1 ---


Featurizing p:   0%|          | 0/50 [00:00<?, ?it/s]

Featurizing q:   0%|          | 0/50 [00:00<?, ?it/s]


--- Final Results ---
   Temp  Context Window  Distinct-1  Distinct-2   MAUVE  Perplexity
0   0.5            1024      0.2327      0.5154  0.0091     19.1120
1   0.5             512      0.2327      0.5154  0.0091     19.1120
2   0.5               1      0.2193      0.4600  0.0078     14.4687
3   1.0            1024      0.2089      0.4492  0.0097     15.2643
4   1.0             512      0.2089      0.4492  0.0097     15.2643
5   1.0               1      0.2193      0.4643  0.0097     15.5810
6   1.5            1024      0.2134      0.4592  0.0041     15.3858
7   1.5             512      0.2134      0.4592  0.0041     15.3858
8   1.5               1      0.2111      0.4629  0.0061     14.8746


In [11]:
# Define the prompts to be used for qualitative examples
sample_prompts = [
    "The Battle of Tassafaronga was a naval battle that took place on November 30 , 1942 , between the United States Navy and the Imperial Japanese Navy .",
    "An enzyme is a protein that acts as a biological catalyst . Catalysts accelerate chemical reactions .",
    "Jupiter is the fifth planet from the Sun and the largest in the Solar System ."
]

In [12]:
print("Samples from BEST Strategy (Highest Diversity)")
print("Configuration: Temp = 0.5, Context Window = 1024\n")

for i, prompt in enumerate(sample_prompts):
    generated_text = generate_contrastive(
        prompt,
        expert_model,
        amateur_model,
        tokenizer,
        amateur_temp=0.5,
        context_window=1024,
        max_new_tokens=60  # Adjust as needed
    )
    print(f"Sample {i+1}")
    print(f"Generated Text: {generated_text}\n")

Samples from BEST Strategy (Highest Diversity)
Configuration: Temp = 0.5, Context Window = 1024

Sample 1
Generated Text: The Battle of Tassafaronga was a naval battle that took place on November 30 , 1942 , between the United States Navy and the Imperial Japanese Navy . The battle was fought off the coast of New Zealand , in the vicinity of Tassafaronga Island , in the South Pacific Ocean .

The battle was fought in the waters between New Zealand and the Solomon Islands , and involved the U.S. Navy's Task Force 58 and the Imperial

Sample 2
Generated Text: An enzyme is a protein that acts as a biological catalyst . Catalysts accelerate chemical reactions .

Catalytic enzymes are used in the synthesis of many important chemicals and are found in almost all living organisms.

Catalytic enzymes are also found in plants and animals.

Catalytic enzymes are classified into two groups:

1. Non-catalytic enzymes

Sample 3
Generated Text: Jupiter is the fifth planet from the Sun and the larges

In [13]:
print("\n" + "="*80 + "\n") # Add a separator for clarity
print("Samples from WORST Strategy (Lowest Diversity)")
print("Configuration: Temp = 1.0, Context Window = 1024\n")

for i, prompt in enumerate(sample_prompts):
    generated_text = generate_contrastive(
        prompt,
        expert_model,
        amateur_model,
        tokenizer,
        amateur_temp=1.0,
        context_window=1024,
        max_new_tokens=60 # Adjust as needed
    )
    print(f"Sample {i+1}")
    print(f"Generated Text: {generated_text}\n")



Samples from WORST Strategy (Lowest Diversity)
Configuration: Temp = 1.0, Context Window = 1024

Sample 1
Generated Text: The Battle of Tassafaronga was a naval battle that took place on November 30 , 1942 , between the United States Navy and the Imperial Japanese Navy . The battle was fought off the coast of New Zealand , in the vicinity of Tassafaronga Island , in the South Pacific Ocean .

The battle was fought in the waters of the South Pacific Ocean , between the Japanese Imperial Navy and the United States Navy . The battle was fought in the

Sample 2
Generated Text: An enzyme is a protein that acts as a biological catalyst . Catalysts accelerate chemical reactions .

Catalysts are used in many different industries . They are used in the production of plastics, fertilizers, and many other products.

Catalysts are used in many different industries . They are used in the production of plastics, fertilizers, and many other products.



Sample 3
Generated Text: Jupiter is the fifth

## Analysis and Discussion

### **Best Performance by Metric 🏆**

The experiments show that no single configuration excels across all metrics, highlighting the inherent trade-offs in text generation.

* **Best for Diversity (Distinct-1 & Distinct-2):** The configuration with **Temperature = 0.5** and a **Context Window of 1024 or 512** produced the most lexically diverse text (`Distinct-1: 0.2327`, `Distinct-2: 0.5154`). A lower temperature on the amateur model appears to encourage a wider range of vocabulary from the expert model.
* **Best for Coherence (Perplexity):** The most coherent and predictable text (lowest perplexity) was surprisingly generated with **Temperature = 0.5** and a **Context Window of 1** (`Perplexity: 14.4687`).
* **Best for Human-likeness (MAUVE):** The highest MAUVE score, which measures the distributional similarity to human text, was achieved at **Temperature = 1.0** for all context window sizes (`MAUVE: 0.0097`).

---

### **Key Trade-Offs and Findings** ↔️

The results clearly illustrate the core principles and dynamics of contrastive decoding.

#### **1. The Coherence vs. Diversity Trade-Off**
This is the central finding. The configuration that was **most diverse** (Temp=0.5, Window=1024) was significantly **less coherent** (Perplexity=19.11) than the configuration that was **most coherent** (Temp=0.5, Window=1), which in turn was less diverse. This demonstrates that steering the model away from generic text can push it in different directions: either towards more varied vocabulary or towards more structured, logical progressions, but rarely both simultaneously.

#### **2. The "Context Window Paradox"**
The most insightful and counter-intuitive result is that the **smallest context window (1) produced the most coherent text**. This phenomenon, which we can call the "Context Window Paradox," occurs because when the amateur model sees only the *last token*, its predictions become extremely generic (e.g., after "the," it will strongly predict common nouns). Subtracting this highly generic prediction forces the expert model to discard the most obvious next word and instead select a token that is more contextually specific and logical, thereby improving local coherence.

#### **3. The MAUVE "Sweet Spot"**
The MAUVE score peaked at a moderate temperature of 1.0. This suggests it found a "sweet spot" that balances the trade-offs. The generations at `Temp=1.0` were likely not as repetitive as those at `Temp=0.5` but not as potentially erratic as those at `Temp=1.5`. It's important to note, however, that **all the MAUVE scores are very low** (close to 0), indicating that the generated text distributions were all significantly different from the human-written reference text.

---

### **Connection to Original Paper's Concepts** 📜

The empirical results align perfectly with the theoretical foundations of the contrastive decoding paper by Li et al. (2022).

* **Validation of Core Mechanism:** The paper introduced contrastive decoding to solve the problem of "generic" and repetitive text. The results empirically validate this by showing how manipulating the amateur model's `temperature` and `context_window` directly controls the trade-off between diversity and coherence.
* **Existence of a "Sweet Spot":** The paper argues that the goal is not to maximize any single metric but to find a desirable balance. The MAUVE results, which peak at the moderate temperature of 1.0, support this idea by identifying a configuration that provides the best compromise between the competing pressures of coherence and diversity.
* **Demonstration of the Contrastive Effect:** The "Context Window Paradox" finding provides a powerful and practical demonstration of how the contrastive mechanism works. By severely handicapping the amateur model, you maximize the "contrast," forcing the expert model to produce highly coherent, non-obvious text, which is the exact behavior the paper aimed to achieve.